# 04. レポートの作成

`data/processed/<report_ymd>/` の `market/` (01) と `factor/` (03) から、図 (PNG)・表 (CSV)・パワポを作り、`data/processed/<report_ymd>/report/` に保存して表示します。片方しかなければ、ある方だけ作ります。
パラメータを変えて、すべて実行してください。

- 強調する期間 (リターンの期間) は、省略時は `report_ymd` の前日を終点とする 1 週間です (`configs/weekly_report.yaml`)。
- 決算・経済指標のカレンダーは `report_ymd` 以降です。
- 市況の図は `configs/weekly_slides.yaml` に従って `templates/example.pptx` に差し込みます。

In [ ]:
report_ymd = 20261125  # レポートの日付 (data/processed/<日付>/)
# 強調する期間の最初と最後の日 (YYYYMMDD)。None なら report_ymd の前日を終点とする 1 週間
period_from_ymd = None
period_to_ymd = None
make_slides = True  # 市況の図をパワポのテンプレートに差し込んだファイルも作る

In [ ]:
# em_market のコードを変更したとき、カーネルを再起動せずに反映する
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display

from em_market.paths import processed_dir
from em_market.reports.factor import FactorResults, draw_factor_figures, factor_period
from em_market.reports.slides import SlidesConfig, build_slides
from em_market.reports.weekly import WeeklyData, WeeklyReport, WeeklyReportConfig

report_date = pd.Timestamp(str(report_ymd))
weekly = WeeklyReportConfig.load()
if period_from_ymd is None or period_to_ymd is None:
    period_from_ymd, period_to_ymd = factor_period(
        report_date, weekly.lookback_days, weekly.data_lag_days
    )
market_dir, factor_dir, output_dir = (
    processed_dir(report_date, kind) for kind in ("market", "factor", "report")
)
print(f"対象期間 {period_from_ymd} ～ {period_to_ymd}｜保存先 {output_dir}")

## 市況の図

In [ ]:
if market_dir.exists():
    # リターンは対象期間の前日の値を基準にする
    report = WeeklyReport(
        base_date=report_date,
        config=weekly,
        output_dir=output_dir,
        return_period=(
            pd.Timestamp(str(period_from_ymd)) - pd.Timedelta(days=1),
            pd.Timestamp(str(period_to_ymd)),
        ),
    )
    market_files = report.run(WeeklyData.load(market_dir))
else:
    market_files = []
    print(f"{market_dir} がないので市況の図は作りません (01 を先に実行)")

## ファクターの図

In [ ]:
if factor_dir.exists():
    factor_files = draw_factor_figures(
        FactorResults.load(factor_dir), period_from_ymd, period_to_ymd, output_dir
    )
else:
    factor_files = []
    print(f"{factor_dir} がないのでファクターの図は作りません (03 を先に実行)")

## パワポ

In [ ]:
config = SlidesConfig.load()
if not (make_slides and market_files):
    print("パワポは作りません")
elif not config.template.exists():
    print(f"テンプレートがないのでパワポは作りません: {config.template}")
else:
    pptx = build_slides(
        output_dir, output_dir / config.output_name.format(date=report_date), config
    )
    print(f"Saved: {pptx}")

## 図の確認

In [ ]:
for path in sorted(output_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=path, width=1000))